In [2]:
import numpy as np
import pandas as pd
import torch
import os
import random
import pyarrow as pa
from scipy import sparse
from pathlib import Path

In [3]:
# =========================
# Step 1 · 环境与依赖初始化（PyTorch 多 GPU 基座）
# =========================

# ---- 标准库 ----
import os
import re
import math
import json
import time
import random
from pathlib import Path
from typing import Optional, Tuple, List

# ---- 科学计算 / 数据 IO ----
import numpy as np
import pandas as pd

# Parquet 要求：使用 fastparquet
# pip install fastparquet
# （若需要更高性能也可选 pyarrow，但本项目统一 fastparquet）
PARQUET_ENGINE = "fastparquet"

# 稀疏矩阵（用于与已有管线兼容的三元组落盘；随后将逐步转换为 PyTorch 稀疏）
from scipy import sparse

# ---- PyTorch 栈 ----
import torch
import torch.nn as nn
import torch.nn.functional as F

# 可选：分布式 / 多 GPU。Notebook 推荐 DataParallel；脚本/集群可用 DDP（torchrun）
import torch.distributed as dist

# 提升 matmul 的 float32 精度/吞吐（PyTorch 2.0+）
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# cudnn benchmark 有益于卷积型工作负载；对本项目影响不大，但开启通常无害
torch.backends.cudnn.benchmark = True

# ---- 近邻搜索（后续 Step 7+ 用）----
# 推荐 GPU 版：pip install faiss-gpu==1.7.4
# CPU 版：pip install faiss-cpu==1.7.4
try:
    import faiss  # noqa
    _FAISS_AVAILABLE = True
except Exception:
    _FAISS_AVAILABLE = False

# ---- 通用路径 ----
TMP_DIR = Path("./tmp")  # 统一的中间件目录（你要求）
TMP_DIR.mkdir(exist_ok=True)

# ---- 随机种子 ----
def set_seed(seed: int = 2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(2025)

# ---- GPU/分布式环境工具 ----
def gpu_summary():
    n = torch.cuda.device_count()
    if n == 0:
        print("[GPU] No CUDA device found. Running on CPU.")
        return
    print(f"[GPU] CUDA devices: {n}")
    for i in range(n):
        prop = torch.cuda.get_device_properties(i)
        print(f"  - GPU{i}: {prop.name}, {prop.total_memory/1024**3:.1f} GB, SMs={prop.multi_processor_count}")

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def use_all_gpus_for_module(module: nn.Module) -> nn.Module:
    """
    Notebook/单进程场景：用 DataParallel 覆盖所有 GPU。
    如使用 torchrun/Slurm 多进程 DDP，请在外部初始化后改用 DDP 包装。
    """
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        return nn.DataParallel(module)  # 简便多GPU
    return module

def maybe_init_ddp(backend: str = "nccl"):
    """
    仅当你用 torchrun 启动时才会生效；Notebook 默认不做 DDP 初始化。
    export MASTER_ADDR=... MASTER_PORT=... WORLD_SIZE=... RANK=...
    torchrun --nproc_per_node=NUM_GPUS your_script.py
    """
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        if not dist.is_initialized():
            dist.init_process_group(backend=backend)
            local_rank = int(os.environ.get("LOCAL_RANK", 0))
            torch.cuda.set_device(local_rank)
            print(f"[DDP] Initialized. rank={dist.get_rank()}/{dist.get_world_size()}, local_rank={local_rank}")
    else:
        # Notebook 情况：不启 DDP
        pass

gpu_summary()
print(f"[Env] TMP_DIR = {TMP_DIR.resolve()}")

# ---- Parquet / 稀疏 CSR 三元组 读写工具（与历史管线兼容） ----
def save_parquet_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, engine=PARQUET_ENGINE, index=False)

def load_parquet_df(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path, engine=PARQUET_ENGINE)

def save_csr_as_triplets(mat: sparse.csr_matrix, path: Path, dtype=np.float32):
    coo = mat.tocoo()
    pdf = pd.DataFrame({
        "row": coo.row.astype(np.int32),
        "col": coo.col.astype(np.int32),
        "val": coo.data.astype(dtype),
    })
    save_parquet_df(pdf, path)

def load_csr_from_triplets(path: Path, shape: Tuple[int, int], dtype=np.float32) -> sparse.csr_matrix:
    pdf = load_parquet_df(path)
    coo = sparse.coo_matrix((pdf["val"].astype(dtype), (pdf["row"], pdf["col"])), shape=shape, dtype=dtype)
    return coo.tocsr()

# ---- Torch 稀疏工具（后续步骤会逐步改为 torch.sparse 格式进行 GPU 计算）----
def csr_to_torch_coo(mat: sparse.csr_matrix, device: torch.device = None, dtype: torch.dtype = torch.float32):
    mat = mat.tocoo()
    indices = torch.tensor(np.vstack([mat.row, mat.col]), dtype=torch.long, device=device)
    values  = torch.tensor(mat.data, dtype=dtype, device=device)
    shape   = torch.Size(mat.shape)
    return torch.sparse_coo_tensor(indices, values, shape, device=device, dtype=dtype).coalesce()

def torch_row_l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    对 dense 或稀疏的行做 L2 归一（对稀疏COO：只缩放 values）。
    这里先提供 dense 版本；后续在具体步骤对稀疏做定制归一。
    """
    if x.is_sparse:
        # 稀疏归一在后续针对性实现；此处仅放占位说明
        raise NotImplementedError("Sparse L2 normalize will be implemented in the graph-specific steps.")
    norm = torch.linalg.norm(x, dim=1, keepdim=True).clamp_min(eps)
    return x / norm

print("[Step 1] PyTorch 多 GPU 环境初始化完成。后续步骤将沿用以上工具与路径。")


[GPU] CUDA devices: 2
  - GPU0: NVIDIA RTX 6000 Ada Generation, 47.5 GB, SMs=142
  - GPU1: NVIDIA RTX 6000 Ada Generation, 47.5 GB, SMs=142
[Env] TMP_DIR = /home/koyo/workspace/recsys/tmp
[Step 1] PyTorch 多 GPU 环境初始化完成。后续步骤将沿用以上工具与路径。


In [4]:
# =========================
# Step 2 · Params: I/O + 预处理（保持结构，落盘到 TMP_DIR）
# =========================

from pathlib import Path

# 输入数据路径
DATA_CSV_PATH = Path("./data/metadata_merged.csv")

# 构建 text_all 的原始列（按需要可增/减）
TEXT_COLS = ["Title", "Subtitle", "Description", "Slug"]

# 标签列（CSV中的原始列名）
TAG_COL = "Tags"

# 主键列（数据集ID）
ID_COL = "Id"

# 可能存在的时间列（择优使用）
CREATED_COL_CANDIDATES     = ["CreationDate_dt", "CreationDate"]
LAST_ACTIVE_COL_CANDIDATES = ["LastActivityDate_dt", "LastActivityDate"]

# 文本标准化（轻量）
TEXT_LOWER      = True
TEXT_STRIP_HTML = True   # 粗略移除 <...> 标记
TEXT_DROP_URL   = True   # 粗略移除 http(s) URL

# 保存文件名
DOC_CLEAN_PATH = TMP_DIR / "doc_clean.parquet"
INDEX_MAP_PATH = TMP_DIR / "index_map.parquet"


In [6]:
# =========================
# Step 2 · Execute: 读取CSV→清洗→合列text_all→标签解析→落盘
# =========================

import re
import pandas as pd
import numpy as np

# 1) 读取
df = pd.read_csv(DATA_CSV_PATH, low_memory=False)
print(f"[INFO] 读入: {DATA_CSV_PATH}  shape={df.shape}")

# 2) 选择时间列（有 *_dt 优先）
def _pick_first_exist(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

created_col = _pick_first_exist(df.columns, CREATED_COL_CANDIDATES)
lastact_col = _pick_first_exist(df.columns, LAST_ACTIVE_COL_CANDIDATES)

# 3) 规范 Id
if ID_COL not in df.columns:
    raise ValueError(f"未找到主键列: {ID_COL}")
# 若有重复 Id，保留第一条
df = df.drop_duplicates(subset=[ID_COL], keep="first").reset_index(drop=True)
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")

# 4) 合列 text_all
def _safe_str(x):
    if pd.isna(x): return ""
    return str(x)

text_pieces = []
for col in TEXT_COLS:
    if col in df.columns:
        text_pieces.append(df[col].map(_safe_str))
if text_pieces:
    text_all = text_pieces[0]
    for p in text_pieces[1:]:
        text_all = text_all + " " + p
else:
    text_all = pd.Series([""] * len(df), dtype=str)

# 轻量清洗：去HTML尖括号、URL、小写化
if TEXT_STRIP_HTML:
    text_all = text_all.str.replace(r"<[^>]+>", " ", regex=True)
if TEXT_DROP_URL:
    text_all = text_all.str.replace(r"https?://\S+", " ", regex=True)
if TEXT_LOWER:
    text_all = text_all.str.lower()

# 5) 解析标签 → 存储为逗号分隔字符串（兼容 fastparquet）
def _parse_tags(x: str):
    if pd.isna(x) or not isinstance(x, str): return ""
    # 拆分逗号，去空格，小写；去除空tag
    tags = [t.strip().lower() for t in x.split(",")]
    tags = [t for t in tags if t]
    return ",".join(tags)

if TAG_COL in df.columns:
    tag_str = df[TAG_COL].map(_parse_tags)
else:
    tag_str = pd.Series([""] * len(df), dtype=str)

# 6) 规范化时间
def _to_dt_safe(x):
    try:
        return pd.to_datetime(x, errors="coerce")
    except Exception:
        return pd.NaT

created_at = _to_dt_safe(df[created_col]) if created_col else pd.Series([pd.NaT]*len(df))
last_active_at = _to_dt_safe(df[lastact_col]) if lastact_col else pd.Series([pd.NaT]*len(df))

# 7) 组装 doc_df（保持与后续步骤对齐）
doc_df = pd.DataFrame({
    "doc_idx": np.arange(len(df), dtype=np.int64),
    "Id":      df[ID_COL].astype("Int64"),
    "text_all": text_all.fillna(""),
    "tag_str":  tag_str.fillna(""),
    "created_at": created_at,
    "last_active_at": last_active_at,
})

# 8) 基础统计
n_docs = len(doc_df)
docs_with_tags = int((doc_df["tag_str"].str.len() > 0).sum())
avg_words = float(doc_df["text_all"].str.split().map(len).replace(0, np.nan).mean())
print(f"[INFO] 文档数: {n_docs:,}")
print(f"[INFO] 含标签文档数/占比: {docs_with_tags:,} / {n_docs:,} ({docs_with_tags/n_docs:.1%})")
print(f"[INFO] 平均文本词数(粗略): {avg_words:.2f}")

# 9) 落盘（Parquet / fastparquet）
save_parquet_df(doc_df, DOC_CLEAN_PATH)
index_map = doc_df[["doc_idx","Id"]].copy()
save_parquet_df(index_map, INDEX_MAP_PATH)
print(f"[OK] 已保存：\n  - {DOC_CLEAN_PATH}\n  - {INDEX_MAP_PATH}")


[INFO] 读入: data/metadata_merged.csv  shape=(521735, 31)
[INFO] 文档数: 521,735
[INFO] 含标签文档数/占比: 214,603 / 521,735 (41.1%)
[INFO] 平均文本词数(粗略): 41.41
[OK] 已保存：
  - tmp/doc_clean.parquet
  - tmp/index_map.parquet


In [7]:
# Quick sanity checks for Step 2 outputs
from pathlib import Path
import pandas as pd
import numpy as np

TMP_DIR = Path("./tmp")

doc = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
idx = pd.read_parquet(TMP_DIR / "index_map.parquet", engine="fastparquet")

print(f"[CHECK] doc_clean shape={doc.shape}, index_map shape={idx.shape}")
print("[CHECK] columns:", list(doc.columns))

# 1) 前3行预览（截断文本方便查看）
preview = doc[["doc_idx","Id","tag_str","created_at","last_active_at","text_all"]].head(3).copy()
preview["text_all"] = preview["text_all"].str.slice(0, 120) + "..."
print("\n[PREVIEW]\n", preview.to_string(index=False))

# 2) 唯一性与取值范围
ok_unique = doc["doc_idx"].is_unique
ok_range  = (doc["doc_idx"].min()==0) and (doc["doc_idx"].max()==len(doc)-1)
dup_id    = doc["Id"].duplicated().sum()
print(f"\n[CHECK] doc_idx unique={ok_unique}, contiguous={ok_range}, duplicated Ids={dup_id}")

# 3) 关键列缺失情况
na_stats = doc[["Id","text_all","tag_str","created_at","last_active_at"]].isna().sum()
print("\n[NA counts]\n", na_stats)

# 4) 文本长度分布（词数）
tok_len = doc["text_all"].fillna("").str.split().map(len)
print("\n[TEXT LEN] mean/median/min/max = ",
      f"{tok_len.mean():.2f}/{tok_len.median()}/{tok_len.min()}/{tok_len.max()}")

# 5) 标签覆盖与Top-15标签（若有tag_str）
has_tag = (doc["tag_str"].str.len() > 0)
print(f"\n[TAGS] docs_with_tags={has_tag.sum():,} / {len(doc):,} ({has_tag.mean():.1%})")
if has_tag.any():
    tags_exploded = doc.loc[has_tag, "tag_str"].str.split(",").explode().str.strip()
    top_tags = tags_exploded.value_counts().head(15)
    print("\n[TAGS] top-15\n", top_tags.to_string())

# 6) index_map与doc对齐性
merged = idx.merge(doc[["doc_idx","Id"]], on=["doc_idx","Id"], how="outer", indicator=True)
mismatch = (merged["_merge"] != "both").sum()
print(f"\n[CHECK] index_map alignment mismatches = {mismatch}")

print("\n[OK] Step 2 sanity checks completed.")


[CHECK] doc_clean shape=(521735, 6), index_map shape=(521735, 2)
[CHECK] columns: ['doc_idx', 'Id', 'text_all', 'tag_str', 'created_at', 'last_active_at']

[PREVIEW]
  doc_idx  Id                                      tag_str          created_at last_active_at                                                                                                                    text_all
       0   6 computer science,demographics,social science 2015-07-18 00:51:12     2018-02-05 2013 american community survey find insights in the 2013 american community survey the [american community survey](  is ...
       1   7      internet,linguistics,online communities 2015-08-04 23:59:00     2018-02-06 may 2015 reddit comments get personal with a dataset of comments from may 2015 recently reddit released [an enormous dat...
       2   8              atmospheric science,environment 2015-08-18 21:53:00     2018-01-31 ocean ship logbooks (1750-1850) explore changing climatology with data from early shippin

In [10]:
# =========================
# Step 3 · Params: Tag 视图（GPU）—— D–T Binary / TF-IDF / PPMI
# =========================

from pathlib import Path

# 读写目录（统一）
TMP_DIR = Path("./tmp")

# 词表过滤
MIN_DF_TAG = 10        # 至少在多少文档中出现的标签才保留
MAX_TAGS   = None      # 可设置为 int 做顶部截断；None 表示不截断

# TF-IDF 参数（对 Tag 语境，tf=1，idf 为平滑IDF）
TFIDF_IDF_SMOOTH = True        # True: idf = log((N+1)/(df+1)) + 1; False: idf = log(N/df)
ROW_NORM_TFIDF   = "l2"         # {"l2","l1",None}

# PPMI 参数（对非零项计算 pmi = log( total / (row_deg * col_deg) ) ，再截断为正）
PPMI_EPS         = 1e-12
ROW_NORM_PPMI    = "l2"         # {"l2","l1",None}

# 多 GPU 切分（对非零条目切块并行）
USE_ALL_GPUS = True             # 尽量使用所有 GPU
CHUNKS_PER_GPU = 1              # 每块大小由 nnz / (num_gpus*chunks_per_gpu) 决定


In [15]:
# =========================
# Step 4 · Params: Text 视图（GPU）—— D–W BM25
# =========================

from pathlib import Path

# 统一TMP路径（沿用你的写法）
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— 词表筛选 —— 
MIN_DF_W        = 200       # 词在多少文档中出现才保留（与之前保持一致）
MAX_DF_RATIO_W  = 0.50      # 丢弃出现在>50%文档中的“过常见词”
KEEP_TOP_W      = None      # 也可额外设定保留Top-N高df词，None表示不额外截断

# —— 分词 ——（轻量可控；与原先一致的英文token思路）
TOKEN_PATTERN   = r"[a-z0-9_]+"
MIN_TOKEN_LEN   = 2
TO_LOWER        = True

# 可选停用词（保持简洁；你有更全词表也可替换）
STOPWORDS = {
    "the","and","to","of","in","for","on","with","a","is","this","that",
    "it","as","from","by","an","or","be","are","at","we","you","your",
}

# —— BM25 参数 ——（与 IR 常用默认一致）
BM25_K1 = 1.5
BM25_B  = 0.75

# 行归一（便于后续图扩散）
ROW_NORM_BM25 = "l2"   # {"l2","l1",None}

# —— 性能参数 —— 
CHUNK_DOCS   = 50_000  # CSV→文本→分词→计数的CPU流式块
USE_ALL_GPUS = True
CHUNKS_PER_GPU = 1     # BM25权重计算按nnz切分


In [16]:
# =========================
# Step 3 · Execute: 在 GPU 构建 D–T（三个加权）→ Parquet 落盘
# =========================

import pandas as pd
import numpy as np
import torch
from scipy import sparse

# ---- 1) 读取 Step 2 输出，并解析标签 → 构建 Tag 词表 ----
doc_df = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
N = len(doc_df)

# 解析 tag_str 为列表并展开
tags_series = doc_df["tag_str"].fillna("")
has_tag = tags_series.str.len() > 0

exploded = (
    tags_series[has_tag]
    .str.split(",")
    .explode()
    .str.strip()
    .str.lower()
)
exploded = exploded[exploded != ""]  # 去空

# 统计 df（每个标签在多少文档出现）
# 为确保是“文档级”计数，先拼出 (doc_idx, tag) 对再去重
pairs = pd.DataFrame({
    "doc_idx": doc_df.loc[exploded.index, "doc_idx"].to_numpy(dtype=np.int64),
    "tag": exploded.to_numpy()
})
pairs = pairs.drop_duplicates()

df_counts = pairs.groupby("tag")["doc_idx"].nunique().sort_values(ascending=False)
# 过滤 min_df
df_counts = df_counts[df_counts >= MIN_DF_TAG]
if MAX_TAGS is not None:
    df_counts = df_counts.head(MAX_TAGS)

id2tag = df_counts.index.to_numpy()
T = len(id2tag)
tag2id = {t:i for i,t in enumerate(id2tag)}

print(f"[TAG] kept tags = {T} (min_df >= {MIN_DF_TAG})")

# 保存词表
tag_vocab = pd.DataFrame({
    "tid": np.arange(T, dtype=np.int32),
    "tag": id2tag,
    "df":  df_counts.to_numpy(dtype=np.int64)
})
tag_vocab_path = TMP_DIR / "tag_vocab.parquet"
tag_vocab.to_parquet(tag_vocab_path, engine="fastparquet", index=False)
print(f"[TAG] vocab saved: {tag_vocab_path.as_posix()}")

# ---- 2) 构建 D–T Binary COO（CPU） ----
# 仅保留在词表中的 tag
pairs = pairs[pairs["tag"].isin(tag2id)]
pairs["tid"] = pairs["tag"].map(tag2id).astype(np.int32)
rows_np = pairs["doc_idx"].to_numpy(dtype=np.int64)
cols_np = pairs["tid"].to_numpy(dtype=np.int32)
nnz = len(rows_np)
print(f"[TAG] D–T nnz = {nnz:,}, density = {nnz/(N*max(T,1)):.6e}")

# ---- 3) 转到 GPU，并行计算 TF-IDF 与 PPMI 的“值向量” ----
device_count = torch.cuda.device_count() if USE_ALL_GPUS and torch.cuda.is_available() else 0
devices = [f"cuda:{i}" for i in range(device_count)] if device_count>0 else ["cpu"]
print(f"[GPU] devices used for weighting: {devices}")

# a) 公共统计量（GPU上/或CPU上均可）
df_t = tag_vocab["df"].to_numpy(dtype=np.int64)   # col 度（即 df）
row_deg = np.bincount(rows_np, minlength=N).astype(np.int64)  # 每个 doc 的 kept 标签数

# 转 torch
row_idx_all = torch.from_numpy(rows_np)
col_idx_all = torch.from_numpy(cols_np)
row_deg_t   = torch.from_numpy(row_deg)
col_deg_t   = torch.from_numpy(df_t)
total_pairs = torch.tensor(float(nnz))

# b) 按设备分块
num_chunks = max(1, (device_count or 1) * max(1, CHUNKS_PER_GPU))
chunk_bounds = np.linspace(0, nnz, num_chunks+1, dtype=np.int64)

def _compute_values_chunk(si, ei, dev: str):
    # 输入：全量 row_idx_all/col_idx_all/row_deg_t/col_deg_t
    r = row_idx_all[si:ei].to(dev, non_blocking=True)
    c = col_idx_all[si:ei].to(dev, non_blocking=True)
    row_deg_on_dev = row_deg_t.to(dev)      # 做一次
    col_deg_on_dev = col_deg_t.to(dev)      # 做一次
    rd = row_deg_on_dev[r]                  # 直接在 dev 上索引
    cd = col_deg_on_dev[c]

    # TF-IDF（tf=1）：idf = log((N+1)/(df+1)) + 1  或 log(N/df)
    if TFIDF_IDF_SMOOTH:
        idf = torch.log((torch.tensor(N+1.0, device=dev) / (cd.to(torch.float32)+1.0))) + 1.0
    else:
        idf = torch.log(torch.tensor(float(N), device=dev) / cd.to(torch.float32).clamp_min(1.0))
    tfidf_vals = idf  # tf=1

    # PPMI：pmi = log( total / (r_d * c_t) )
    pmi = torch.log(total_pairs.to(dev) / (rd.to(torch.float32)*cd.to(torch.float32)).clamp_min(PPMI_EPS))
    ppmi_vals = torch.clamp(pmi, min=0.0)

    return tfidf_vals.detach().to("cpu"), ppmi_vals.detach().to("cpu")

# 并行计算
tfidf_list, ppmi_list = [], []
for i in range(num_chunks):
    si, ei = int(chunk_bounds[i]), int(chunk_bounds[i+1])
    dev = devices[i % len(devices)]
    tvi, pvi = _compute_values_chunk(si, ei, dev)
    tfidf_list.append(tvi)
    ppmi_list.append(pvi)

tfidf_vals = torch.cat(tfidf_list).numpy()
ppmi_vals  = torch.cat(ppmi_list).numpy()

# ---- 4) 行归一（L2/L1/None），随后落盘三元组 ----
def _row_normalize_values(rows, vals, num_rows, norm_type: str):
    if norm_type is None:
        return vals
    rows_t = torch.from_numpy(rows).long().to("cuda" if torch.cuda.is_available() else "cpu")
    vals_t = torch.from_numpy(vals).to(rows_t.device, dtype=torch.float32)

    if norm_type.lower() == "l2":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        # 累加平方和
        accum.index_add_(0, rows_t, vals_t*vals_t)
        denom = torch.sqrt(torch.clamp(accum, min=1e-12))
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    elif norm_type.lower() == "l1":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        accum.index_add_(0, rows_t, vals_t.abs())
        denom = torch.clamp(accum, min=1e-12)
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    else:
        return vals

tfidf_vals_norm = _row_normalize_values(rows_np, tfidf_vals, N, ROW_NORM_TFIDF)
ppmi_vals_norm  = _row_normalize_values(rows_np, ppmi_vals,  N, ROW_NORM_PPMI)

# Binary 的值恒为 1；可按需要做行归一（通常不需要），这里保持原样
bin_vals = np.ones_like(rows_np, dtype=np.float32)

# ---- 5) 保存三份矩阵：DT_bin / DT_tfidf / DT_ppmi ----
def _save_triplets(rows, cols, vals, shape, path: Path):
    coo = sparse.coo_matrix((vals, (rows, cols)), shape=shape, dtype=np.float32)
    # 复用 Step 1 的工具（如未定义，可直接 to_parquet）
    from pandas import DataFrame
    pdf = DataFrame({
        "row": coo.row.astype(np.int32),
        "col": coo.col.astype(np.int32),
        "val": coo.data.astype(np.float32),
    })
    pdf.to_parquet(path, engine="fastparquet", index=False)

DT_bin_path   = TMP_DIR / "DT_bin.parquet"
DT_tfidf_path = TMP_DIR / "DT_tfidf.parquet"
DT_ppmi_path  = TMP_DIR / "DT_ppmi.parquet"

shape = (N, T)
_save_triplets(rows_np, cols_np, bin_vals,         shape, DT_bin_path)
_save_triplets(rows_np, cols_np, tfidf_vals_norm,  shape, DT_tfidf_path)
_save_triplets(rows_np, cols_np, ppmi_vals_norm,   shape, DT_ppmi_path)

print("[Step 3] Saved:",
      DT_bin_path.as_posix(), DT_tfidf_path.as_posix(), DT_ppmi_path.as_posix())


[TAG] kept tags = 394 (min_df >= 10)
[TAG] vocab saved: tmp/tag_vocab.parquet
[TAG] D–T nnz = 445,426, density = 2.166853e-03
[GPU] devices used for weighting: ['cuda:0', 'cuda:1']
[Step 3] Saved: tmp/DT_bin.parquet tmp/DT_tfidf.parquet tmp/DT_ppmi.parquet


In [17]:
# =========================
# Step 4 · Params: Text 视图（GPU）—— D–W BM25
# =========================

from pathlib import Path

# 统一TMP路径（沿用你的写法）
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— 词表筛选 —— 
MIN_DF_W        = 200       # 词在多少文档中出现才保留（与之前保持一致）
MAX_DF_RATIO_W  = 0.50      # 丢弃出现在>50%文档中的“过常见词”
KEEP_TOP_W      = None      # 也可额外设定保留Top-N高df词，None表示不额外截断

# —— 分词 ——（轻量可控；与原先一致的英文token思路）
TOKEN_PATTERN   = r"[a-z0-9_]+"
MIN_TOKEN_LEN   = 2
TO_LOWER        = True

# 可选停用词（保持简洁；你有更全词表也可替换）
STOPWORDS = {
    "the","and","to","of","in","for","on","with","a","is","this","that",
    "it","as","from","by","an","or","be","are","at","we","you","your",
}

# —— BM25 参数 ——（与 IR 常用默认一致）
BM25_K1 = 1.5
BM25_B  = 0.75

# 行归一（便于后续图扩散）
ROW_NORM_BM25 = "l2"   # {"l2","l1",None}

# —— 性能参数 —— 
CHUNK_DOCS   = 50_000  # CSV→文本→分词→计数的CPU流式块
USE_ALL_GPUS = True
CHUNKS_PER_GPU = 1     # BM25权重计算按nnz切分


In [18]:
# =========================
# Step 4 · Execute: 读取text_all→建词表→构建 D–W 计数→GPU上BM25加权→行归一→落盘
# =========================

import re
import numpy as np
import pandas as pd
import torch
from scipy import sparse

# ---------- 0) 读入清洗后的文档 ----------
doc_df = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
N = len(doc_df)
print(f"[TEXT] docs={N:,}")

# ---------- 1) 首遍：扫描文本→构建词表(df) ----------
token_re = re.compile(TOKEN_PATTERN)

def tokenize(s: str):
    if not isinstance(s, str):
        return []
    if TO_LOWER:
        s = s.lower()
    toks = token_re.findall(s)
    toks = [t for t in toks if len(t) >= MIN_TOKEN_LEN and t not in STOPWORDS]
    return toks

df_counter = {}            # word -> df
doc_len = np.zeros(N, dtype=np.int32)

for base in range(0, N, CHUNK_DOCS):
    part = doc_df.iloc[base: base+CHUNK_DOCS]
    unique_in_doc = []   # 收集每个doc的唯一token集合 以计 df
    for i, s in zip(part["doc_idx"].to_numpy(), part["text_all"].tolist()):
        toks = tokenize(s)
        doc_len[i] = len(toks)
        if len(toks) == 0:
            unique_in_doc.append(())
            continue
        uniq = set(toks)
        unique_in_doc.append(uniq)
    # 累加 df
    for uniq in unique_in_doc:
        for w in uniq:
            df_counter[w] = df_counter.get(w, 0) + 1

# 构建词表并过滤
vocab_df = pd.DataFrame({"word": list(df_counter.keys()),
                         "df":   list(df_counter.values())})
vocab_df.sort_values("df", ascending=False, inplace=True)
# 过滤 df / max_df_ratio
vocab_df = vocab_df[vocab_df["df"] >= MIN_DF_W]
vocab_df = vocab_df[vocab_df["df"] <= MAX_DF_RATIO_W * N]
if KEEP_TOP_W is not None:
    vocab_df = vocab_df.head(KEEP_TOP_W)
vocab_df.reset_index(drop=True, inplace=True)

id2word = vocab_df["word"].to_numpy()
df_w    = vocab_df["df"].to_numpy(dtype=np.int64)
W = len(id2word)
word2id = {w:i for i,w in enumerate(id2word)}
print(f"[TEXT] raw_vocab={len(df_counter):,} -> kept(min_df>={MIN_DF_W}, max_df<={MAX_DF_RATIO_W*100:.0f}%)={W:,}")

# 落盘 text_vocab
text_vocab = pd.DataFrame({
    "wid": np.arange(W, dtype=np.int32),
    "word": id2word,
    "df": df_w
})
text_vocab_path = TMP_DIR / "text_vocab.parquet"
text_vocab.to_parquet(text_vocab_path, engine="fastparquet", index=False)
print(f"[TEXT] vocab saved: {text_vocab_path.as_posix()}")

# ---------- 2) 二遍：构建 D–W 计数（三元组） ----------
rows, cols, vals = [], [], []    # tf (term frequency)
row_ptr = []                     # 可选：统计nnz/行

for base in range(0, N, CHUNK_DOCS):
    part = doc_df.iloc[base: base+CHUNK_DOCS]
    for i, s in zip(part["doc_idx"].to_numpy(), part["text_all"].tolist()):
        toks = tokenize(s)
        # 仅保留词表中的词
        tids = [word2id[t] for t in toks if t in word2id]
        if not tids:
            continue
        # 统计该doc的tf
        # 为了速度，使用小字典计数
        cnt = {}
        for t in tids:
            cnt[t] = cnt.get(t, 0) + 1
        # 追加三元组
        for t, f in cnt.items():
            rows.append(i)
            cols.append(t)
            vals.append(f)

rows_np = np.asarray(rows, dtype=np.int64)
cols_np = np.asarray(cols, dtype=np.int32)
tfs_np  = np.asarray(vals, dtype=np.float32)
nnz = len(rows_np)
print(f"[TEXT] D–W nnz={nnz:,}, density={nnz/(N*max(W,1)):.6e}")

# ---------- 3) 在GPU上计算 BM25 权重 ----------
device_count = torch.cuda.device_count() if USE_ALL_GPUS and torch.cuda.is_available() else 0
devices = [f"cuda:{i}" for i in range(device_count)] if device_count>0 else ["cpu"]
print(f"[GPU] devices used for BM25: {devices}")

avgdl = float(doc_len.mean()) if doc_len.mean() > 0 else 1.0

row_idx_all = torch.from_numpy(rows_np)
col_idx_all = torch.from_numpy(cols_np)
tf_all      = torch.from_numpy(tfs_np)

row_len_t = torch.from_numpy(doc_len)     # dl per doc
df_t      = torch.from_numpy(df_w)        # df per word
N_t       = torch.tensor(float(N))

num_chunks = max(1, (device_count or 1) * max(1, CHUNKS_PER_GPU))
bounds = np.linspace(0, nnz, num_chunks+1, dtype=np.int64)

def _bm25_chunk(si, ei, dev: str):
    r = row_idx_all[si:ei].to(dev, non_blocking=True)
    c = col_idx_all[si:ei].to(dev, non_blocking=True)
    tf = tf_all[si:ei].to(dev, non_blocking=True).to(torch.float32)

    # gather dl/df（在CPU上索引再搬/或直接映射到同设备）
    dl = row_len_t[r.cpu()].to(dev, non_blocking=True).to(torch.float32).clamp_min(1.0)
    df = df_t[c.cpu()].to(dev, non_blocking=True).to(torch.float32).clamp_min(1.0)

    # idf = log( (N - df + 0.5) / (df + 0.5) + 1 )
    idf = torch.log(((N_t.to(dev) - df + 0.5) / (df + 0.5)).clamp_min(1e-12) + 1.0)

    # denom = tf + k1*(1 - b + b*dl/avgdl)
    denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * (dl / avgdl))
    bm25 = idf * (tf * (BM25_K1 + 1.0) / denom.clamp_min(1e-12))

    return bm25.detach().to("cpu")

bm25_vals_list = []
for i in range(num_chunks):
    si, ei = int(bounds[i]), int(bounds[i+1])
    dev = devices[i % len(devices)]
    bm25_vals_list.append(_bm25_chunk(si, ei, dev))

bm25_vals = torch.cat(bm25_vals_list).numpy()

# ---------- 4) 行归一（L2 / L1 / None） ----------
def _row_normalize_values(rows, vals, num_rows, norm_type: str):
    if norm_type is None:
        return vals
    rows_t = torch.from_numpy(rows).long().to("cuda" if torch.cuda.is_available() else "cpu")
    vals_t = torch.from_numpy(vals).to(rows_t.device, dtype=torch.float32)
    if norm_type.lower() == "l2":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        accum.index_add_(0, rows_t, vals_t*vals_t)
        denom = torch.sqrt(torch.clamp(accum, min=1e-12))
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    elif norm_type.lower() == "l1":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        accum.index_add_(0, rows_t, vals_t.abs())
        denom = torch.clamp(accum, min=1e-12)
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    else:
        return vals

bm25_vals_norm = _row_normalize_values(rows_np, bm25_vals, N, ROW_NORM_BM25)

# ---------- 5) 落盘 DW_bm25（三元组） ----------
DW_bm25_path = TMP_DIR / "DW_bm25.parquet"
coo = sparse.coo_matrix((bm25_vals_norm, (rows_np, cols_np)), shape=(N, W), dtype=np.float32)
pd.DataFrame({
    "row": coo.row.astype(np.int32),
    "col": coo.col.astype(np.int32),
    "val": coo.data.astype(np.float32),
}).to_parquet(DW_bm25_path, engine="fastparquet", index=False)

print("[Step 4] Saved:", DW_bm25_path.as_posix())


[TEXT] docs=521,735
[TEXT] raw_vocab=599,004 -> kept(min_df>=200, max_df<=50%)=5,754
[TEXT] vocab saved: tmp/text_vocab.parquet
[TEXT] D–W nnz=8,140,467, density=2.711624e-03
[GPU] devices used for BM25: ['cuda:0', 'cuda:1']
[Step 4] Saved: tmp/DW_bm25.parquet


In [19]:
# =========================
# Step 5 · Params: GPU 随机游走（类型约束 D–X–D → D-only 语料）
# =========================

from pathlib import Path

# 统一路径
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— 随机游走参数（高性能取向，与前面一致） ——
RW_WALKS_PER_DOC    = 10       # 每个可用起点D生成多少条游走
RW_L_DOCS_PER_SENT  = 40       # D-only 序列长度（D个数）
RW_SEED_BASE        = 2025     # 基础随机种子（每轮 __iter__ 会 + 偏移）
RW_AVOID_BACKTRACK  = True     # 禁止 D 立即回到上一个 D
RW_RESTART_PROB     = 0.15     # 以概率回到起点 d0（PPR 风格）
RW_X_DEGREE_POW     = -0.5     # X 侧度数惩罚：乘以 (deg(X))^pow 抑制 hub（如 -0.5）
RW_X_NO_REPEAT_LAST = 1        # 最近访问的 X 在下一步禁用（1=不重复上一个 X）

# —— 多 GPU 使用策略 ——
USE_ALL_GPUS   = True          # 使用所有可用 GPU
SPLIT_SHARDS   = 32            # 将起点划分为多少 shard，轮转到各 GPU 上

# —— 调试/抽样输出（生产环境请置0） ——
RW_INSPECT_SAMPLES = 0         # >0 会打印若干条 D-only 样例（不建议太大）


In [20]:
# =========================
# Step 5 · Execute: 读入 D–T / D–W → 构建 GPU 随机游走生成器（在线）
# =========================

import numpy as np
import pandas as pd
import torch
from scipy import sparse

# ---------- 1) 读入准备 ----------
doc_df    = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet")
text_vocab= pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")

N = len(doc_df)
T = len(tag_vocab)
W = len(text_vocab)

def load_csr_from_triplets(path, shape, dtype=np.float32):
    pdf = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((pdf["val"].astype(dtype), (pdf["row"], pdf["col"])),
                            shape=shape, dtype=dtype)
    return coo.tocsr()

DT_ppmi = load_csr_from_triplets(TMP_DIR / "DT_ppmi.parquet", shape=(N, T))
DW_bm25 = load_csr_from_triplets(TMP_DIR / "DW_bm25.parquet", shape=(N, W))

# ---------- 2) 将 CSR 转为 torch 张量（并在每块 GPU 上各保留一份轻量数据） ----------
def csr_to_torch_rowview(mat: sparse.csr_matrix, device: torch.device):
    """把 CSR 的 (indptr, indices, data) 移到指定 device；用于快速行切片。"""
    mat = mat.tocsr()
    indptr = torch.from_numpy(mat.indptr.astype(np.int64)).to(device)
    indices= torch.from_numpy(mat.indices.astype(np.int64)).to(device)
    data   = torch.from_numpy(mat.data.astype(np.float32)).to(device)
    return indptr, indices, data

def csr_transpose(mat: sparse.csr_matrix) -> sparse.csr_matrix:
    return mat.transpose().tocsr()

devices = []
if torch.cuda.is_available() and USE_ALL_GPUS:
    devices = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
if not devices:
    devices = [torch.device("cpu")]
print(f"[RW] devices: {[str(d) for d in devices]}")

# 为每个 device 构造两视图的 D->X / X->D 访问结构
DX_tag_dev, XD_tag_dev = [], []
DX_txt_dev, XD_txt_dev = [], []

for dev in devices:
    DX_tag_dev.append(csr_to_torch_rowview(DT_ppmi, dev))
    XD_tag_dev.append(csr_to_torch_rowview(csr_transpose(DT_ppmi), dev))
    DX_txt_dev.append(csr_to_torch_rowview(DW_bm25, dev))
    XD_txt_dev.append(csr_to_torch_rowview(csr_transpose(DW_bm25), dev))

# 预先计算 row/col 度数（在 CPU 上，便于快速判断起点）
degD_tag = np.diff(DT_ppmi.indptr)
degD_txt = np.diff(DW_bm25.indptr)

start_tag = np.where(degD_tag > 0)[0].astype(np.int64)
start_txt = np.where(degD_txt > 0)[0].astype(np.int64)

print(f"[RW-tag] starts={len(start_tag):,}, target_walks≈{len(start_tag)*RW_WALKS_PER_DOC:,}, L={RW_L_DOCS_PER_SENT}")
print(f"[RW-text] starts={len(start_txt):,}, target_walks≈{len(start_txt)*RW_WALKS_PER_DOC:,}, L={RW_L_DOCS_PER_SENT}")

# ---------- 3) 工具函数：行邻接访问 + 加权采样 ----------
def row_neighbors(indptr: torch.Tensor, indices: torch.Tensor, data: torch.Tensor, r: torch.Tensor):
    """返回行 r 的 (cols, weights) 视图（不复制）。r 是标量 LongTensor。"""
    a = indptr[r]
    b = indptr[r+1]
    if (b - a).item() <= 0:
        return None, None
    sl = slice(a.item(), b.item())
    return indices[sl], data[sl]

def sample_pos_by_weights(w: torch.Tensor, g: torch.Generator) -> int:
    """给定非负权重 w，按概率采样返回位置（-1 表示失败）。"""
    # 安全处理
    if w is None or w.numel() == 0:
        return -1
    w = torch.clamp(w, min=0)
    s = w.sum()
    if not torch.isfinite(s) or s.item() <= 0:
        return -1
    # 前缀和 + 二分
    cdf = torch.cumsum(w, dim=0)
    u = torch.rand((), generator=g, device=w.device) * cdf[-1]
    pos = torch.searchsorted(cdf, u, right=False).item()
    if pos >= cdf.numel(): pos = cdf.numel()-1
    return pos

# ---------- 4) 在线语料生成器（两视图各一个），支持多 GPU 分片 ----------
class TorchWalkCorpus:
    """
    每次 __iter__：
      1) 重新洗牌起点并分 shard；
      2) 轮转到各 GPU，使用对应的 DX/XD 结构做 D–X–D 随机游走；
      3) 产出 D-only 序列（list[str]）。
    """
    def __init__(self, starts_np: np.ndarray, DX_views, XD_views, view_name: str, base_seed: int):
        self.starts_np = starts_np
        self.DX_views = DX_views     # list of (indptr, indices, data) per device
        self.XD_views = XD_views
        self.view_name = view_name
        self.base_seed = base_seed
        self._iters = 0

    def __len__(self):
        return int(len(self.starts_np) * RW_WALKS_PER_DOC)

    def __iter__(self):
        self._iters += 1
        rng = np.random.default_rng(self.base_seed + 31*self._iters)
        starts = self.starts_np.copy()
        rng.shuffle(starts)

        # 分 shard，轮转设备
        shards = np.array_split(starts, max(1, SPLIT_SHARDS))
        devs = devices

        for shard_id, shard in enumerate(shards):
            dev = devs[shard_id % len(devs)]
            DX = self.DX_views[shard_id % len(devs)]
            XD = self.XD_views[shard_id % len(devs)]
            indptr_D, indices_D, data_D = DX
            indptr_X, indices_X, data_X = XD

            # 每个 shard 使用独立的 torch RNG
            g = torch.Generator(device=dev)
            g.manual_seed(self.base_seed + 7919 * (self._iters + shard_id))

            # 预备 X 侧度数惩罚
            if abs(RW_X_DEGREE_POW) > 1e-12:
                # X 的度 = XD 行度
                x_deg = (indptr_X[1:] - indptr_X[:-1]).to(torch.float32)
                x_factor = torch.clamp(x_deg, min=1.0).pow(RW_X_DEGREE_POW)
            else:
                x_factor = None

            # 遍历 shard 中的起点
            for si, d0 in enumerate(shard):
                d0_t = torch.tensor(int(d0), dtype=torch.long, device=dev)
                for _ in range(RW_WALKS_PER_DOC):
                    seq = [int(d0)]
                    prev_d = None
                    cur_d  = int(d0)
                    last_x = -1  # 最近一次访问的 X（-1 表示无）

                    for _step in range(RW_L_DOCS_PER_SENT-1):
                        # D -> X
                        r = torch.tensor(cur_d, dtype=torch.long, device=dev)
                        x_cols, x_w = row_neighbors(indptr_D, indices_D, data_D, r)
                        if x_cols is None:
                            break

                        # 权重：边权 × X度数惩罚
                        w = x_w.clone()
                        if x_factor is not None:
                            w = w * x_factor[x_cols]

                        # 禁止重复上一个 X
                        if RW_X_NO_REPEAT_LAST > 0 and last_x >= 0:
                            mask = (x_cols == last_x)
                            if mask.any():
                                w[mask] = 0.0

                        pos_x = sample_pos_by_weights(w, g)
                        if pos_x < 0:
                            break
                        x = int(x_cols[pos_x].item())

                        # X -> D
                        # 通过 XD（X 行）找到相邻 D
                        xr = torch.tensor(x, dtype=torch.long, device=dev)
                        d_rows, d_w = row_neighbors(indptr_X, indices_X, data_X, xr)
                        if d_rows is None:
                            break

                        # 禁回头
                        if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.numel() > 1:
                            mask = (d_rows == prev_d)
                            if mask.any():
                                d_w = d_w.clone()
                                d_w[mask] = 0.0

                        pos_d = sample_pos_by_weights(d_w, g)
                        if pos_d < 0:
                            break
                        next_d = int(d_rows[pos_d].item())

                        # 重启
                        if torch.rand((), generator=g, device=dev).item() < RW_RESTART_PROB:
                            next_d = int(d0)

                        seq.append(next_d)
                        prev_d, cur_d = cur_d, next_d
                        last_x = x

                    if len(seq) >= 2:
                        # 产出 D-only 序列（str token）
                        yield [str(s) for s in seq]

# ---------- 5) 实例化两视图的生成器 ----------
tag_corpus_torch  = TorchWalkCorpus(start_tag, DX_tag_dev, XD_tag_dev, view_name="tag",  base_seed=RW_SEED_BASE+11)
text_corpus_torch = TorchWalkCorpus(start_txt, DX_txt_dev, XD_txt_dev, view_name="text", base_seed=RW_SEED_BASE+23)

print(f"[RW] tag ~{len(tag_corpus_torch):,} sentences; text ~{len(text_corpus_torch):,} sentences; L={RW_L_DOCS_PER_SENT}")

# ---------- 6) 可选：抽样打印若干条（便于快速肉眼质检；跑完可以删除） ----------
if RW_INSPECT_SAMPLES > 0:
    import itertools
    print("\n[INSPECT] sample tag sequences:")
    for s in itertools.islice(iter(tag_corpus_torch), RW_INSPECT_SAMPLES):
        print("  ", s[:min(len(s), 20)])
    print("\n[INSPECT] sample text sequences:")
    for s in itertools.islice(iter(text_corpus_torch), RW_INSPECT_SAMPLES):
        print("  ", s[:min(len(s), 20)])

# ---------- 7) 落盘：保存随机游走参数（便于 Step 6 自给自足重建生成器） ----------
pd.DataFrame([{
    "RW_WALKS_PER_DOC": RW_WALKS_PER_DOC,
    "RW_L_DOCS_PER_SENT": RW_L_DOCS_PER_SENT,
    "RW_SEED_BASE": RW_SEED_BASE,
    "RW_AVOID_BACKTRACK": RW_AVOID_BACKTRACK,
    "RW_RESTART_PROB": RW_RESTART_PROB,
    "RW_X_DEGREE_POW": RW_X_DEGREE_POW,
    "RW_X_NO_REPEAT_LAST": RW_X_NO_REPEAT_LAST,
    "USE_ALL_GPUS": USE_ALL_GPUS,
    "SPLIT_SHARDS": SPLIT_SHARDS,
}]).to_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet", index=False)

print("[Step 5] Online corpus builders ready. Params saved:", (TMP_DIR / "rw_params.parquet").as_posix())


[RW] devices: ['cuda:0', 'cuda:1']
[RW-tag] starts=214,585, target_walks≈2,145,850, L=40
[RW-text] starts=416,697, target_walks≈4,166,970, L=40
[RW] tag ~2,145,850 sentences; text ~4,166,970 sentences; L=40
[Step 5] Online corpus builders ready. Params saved: tmp/rw_params.parquet


In [25]:
# 更清晰的链式打印（可运行后删除）
import numpy as np
import pandas as pd
from scipy import sparse
from pathlib import Path

TMP_DIR = Path("./tmp")
doc_df    = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet")
text_vocab= pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")

def load_csr(path, shape, dtype=np.float32):
    df = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((df["val"].astype(dtype), (df["row"], df["col"])), shape=shape, dtype=dtype)
    return coo.tocsr()

DT_ppmi = load_csr(TMP_DIR/"DT_ppmi.parquet", shape=(len(doc_df), len(tag_vocab)))
DW_bm25 = load_csr(TMP_DIR/"DW_bm25.parquet", shape=(len(doc_df), len(text_vocab)))

rw = pd.read_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet").iloc[0]
RW_RESTART_PROB     = float(rw["RW_RESTART_PROB"])
RW_AVOID_BACKTRACK  = bool(rw["RW_AVOID_BACKTRACK"])
RW_X_DEGREE_POW     = float(rw["RW_X_DEGREE_POW"])
RW_X_NO_REPEAT_LAST = int(rw["RW_X_NO_REPEAT_LAST"])

def _row_neighbors_csr(mat, r):
    a,b = mat.indptr[r], mat.indptr[r+1]
    if a==b: return None, None
    return mat.indices[a:b], mat.data[a:b]

def _sample_by_weights(w, rng):
    w = np.asarray(w, dtype=np.float64)
    tot = w.sum()
    if not np.isfinite(tot) or tot<=0: return None
    cdf = np.cumsum(w)
    pos = int(np.searchsorted(cdf, rng.random()*tot, side="left"))
    if pos >= cdf.size: pos = cdf.size-1
    return pos

def print_chain(name, DX, XD, id2x, x_lab, n_samples=3, L_docs=8, seed=123):
    rng = np.random.default_rng(seed)
    starts = np.where(np.diff(DX.indptr) > 0)[0]
    x_deg = np.diff(XD.indptr).astype(np.float64)
    x_factor = np.power(np.maximum(x_deg,1.0), RW_X_DEGREE_POW) if abs(RW_X_DEGREE_POW)>1e-12 else None

    print(f"\n[CHECK-{name}] {n_samples} 条（目标 D 长度≈{L_docs}）")
    for k in range(n_samples):
        d0 = int(rng.choice(starts))
        seqD = [d0]; seqX = []
        prev_d, cur_d, last_x = None, d0, None
        for _ in range(L_docs-1):
            x_cols, x_w = _row_neighbors_csr(DX, cur_d)
            if x_cols is None: break
            w = x_w.copy()
            if x_factor is not None: w *= x_factor[x_cols]
            if RW_X_NO_REPEAT_LAST>0 and last_x is not None:
                w = w.copy(); w[x_cols==last_x] = 0.0
            p = _sample_by_weights(w, rng); 
            if p is None: break
            x = int(x_cols[p]); seqX.append(x)

            d_rows, d_w = _row_neighbors_csr(XD, x)
            if d_rows is None: break
            if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.size>1:
                d_w = d_w.copy(); d_w[d_rows==prev_d] = 0.0
            p2 = _sample_by_weights(d_w, rng)
            if p2 is None: break
            next_d = int(d_rows[p2])

            if rng.random() < RW_RESTART_PROB:
                next_d = d0
            seqD.append(next_d)
            prev_d, cur_d, last_x = cur_d, next_d, x

        # 连续打印：D0 --X[x0]--> D1 --X[x1]--> D2 ...
        chain = f"D({seqD[0]}/Id={int(doc_df.loc[seqD[0],'Id'])})"
        for i, x in enumerate(seqX):
            xname = id2x[x]
            d_id  = int(doc_df.loc[seqD[i+1],"Id"])
            chain += f" --{x_lab}[{xname}]--> D({seqD[i+1]}/Id={d_id})"
        print(f"  示例 #{k+1}\n  {chain}")

print_chain("tag",  DT_ppmi, DT_ppmi.transpose().tocsr(), tag_vocab["tag"].to_numpy(),  "T", n_samples=3, L_docs=8)
print_chain("text", DW_bm25, DW_bm25.transpose().tocsr(), text_vocab["word"].to_numpy(), "W", n_samples=3, L_docs=8)
print("\n[OK] 更清晰的链式打印完成。")



[CHECK-tag] 3 条（目标 D 长度≈8）
  示例 #1
  D(4911/Id=14578) --T[arts and entertainment]--> D(77827/Id=1269722)
  示例 #2
  D(335565/Id=5161079) --T[beginner]--> D(376473/Id=5751643) --T[news]--> D(399727/Id=6139562)
  示例 #3
  D(8369/Id=37754) --T[standardized testing]--> D(460708/Id=7140595)

[CHECK-text] 3 条（目标 D 长度≈8）
  示例 #1
  D(7249/Id=31415) --W[test]--> D(135089/Id=2007600) --W[code]--> D(417767/Id=6437396) --W[official]--> D(504488/Id=7819842) --W[face]--> D(117467/Id=1767210) --W[dataset]--> D(423370/Id=6539906)
  示例 #2
  D(354554/Id=5398358) --W[infectious]--> D(48176/Id=835414) --W[chest]--> D(81804/Id=1317048) --W[45]--> D(345889/Id=5259310) --W[12]--> D(230416/Id=3500693) --W[lyft]--> D(58777/Id=973364) --W[valid]--> D(104202/Id=1585399) --W[tfrecords]--> D(80985/Id=1307594)
  示例 #3
  D(236152/Id=3585148) --W[million]--> D(236152/Id=3585148) --W[ranked]--> D(111755/Id=1688726) --W[2020]--> D(236152/Id=3585148) --W[various]--> D(341366/Id=5251121) --W[trends]--> D(236152/Id=3585148

In [1]:
# =========================
# Step 6 · Params (Fast WS-SGNS)
# =========================

from pathlib import Path

# 统一中间目录（沿用你的设定）
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— SGNS 超参（提速版，显存友好）——
SGNS_DIM        = 256       # 可调到 192 进一步降显存
SGNS_WINDOW     = 5
SGNS_NEGATIVE   = 12        # 如需更快可降到 10
SGNS_EPOCHS     = 3         # 先 3 轮，吞吐提升后可再加
SGNS_LR         = 0.05
SGNS_CLIP_NORM  = 2.0

# 批量（以“正样本对数”为单位）+ 梯度累积
PAIRS_BATCH_SIZE = 245760    # OOM 就调 16384 / 12288
ACCUM_STEPS      = 2         # 有效等价 batch ≈ 49,152

# 语料规模上限（None=全量）。text 视图很大，先限 1.2M 句以控制时长
MAX_SENTS_PER_EPOCH_TAG  = None
MAX_SENTS_PER_EPOCH_TEXT = 1_200_000

# 负采样分布：基于文档度数^0.75
NS_POWER = 0.75

# —— 关键提速：上下文子采样 + GPU 负采样 + 混合精度 —— 
SGNS_CTX_SAMPLES      = 2          # 每个 center 随机取 2 个 context（替代“全部”）
NEG_SAMPLING_DEVICE   = "gpu"      # {"gpu","cpu"}
SGNS_MIXED_PRECISION  = "bf16"     # {"bf16","fp16", None}

# 随机种子基
RNG_BASE_SEED = 2025

# 多 GPU 使用（沿用 DataParallel）
USE_ALL_GPUS = True


In [2]:
# =========================
# Step 6 · Execute (Fast WS-SGNS)
# =========================

import math, gc, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import sparse

# ---------- 0) 加载中间件 ----------
doc_df    = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet")
text_vocab= pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")
rw_params = pd.read_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet").iloc[0]

N = len(doc_df);  T = len(tag_vocab);  W = len(text_vocab)

def load_csr(path, shape, dtype=np.float32):
    df = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((df["val"].astype(dtype), (df["row"], df["col"])), shape=shape, dtype=dtype)
    return coo.tocsr()

DT_ppmi = load_csr(TMP_DIR / "DT_ppmi.parquet", shape=(N, T))
DW_bm25 = load_csr(TMP_DIR / "DW_bm25.parquet", shape=(N, W))

# ---------- 1) 重建在线语料生成器（复用 Step 5 思路） ----------
def csr_to_rowview_torch(mat: sparse.csr_matrix, device: torch.device):
    mat = mat.tocsr()
    indptr = torch.from_numpy(mat.indptr.astype(np.int64)).to(device)
    indices= torch.from_numpy(mat.indices.astype(np.int64)).to(device)
    data   = torch.from_numpy(mat.data.astype(np.float32)).to(device)
    return indptr, indices, data

def csr_T(mat: sparse.csr_matrix) -> sparse.csr_matrix:
    return mat.transpose().tocsr()

devices = []
if torch.cuda.is_available() and USE_ALL_GPUS:
    devices = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
if not devices:
    devices = [torch.device("cpu")]
print(f"[SGNS] using devices: {[str(d) for d in devices]}")

DX_tag_dev = []; XD_tag_dev = []
DX_txt_dev = []; XD_txt_dev = []
for dev in devices:
    DX_tag_dev.append(csr_to_rowview_torch(DT_ppmi, dev))
    XD_tag_dev.append(csr_to_rowview_torch(csr_T(DT_ppmi), dev))
    DX_txt_dev.append(csr_to_rowview_torch(DW_bm25, dev))
    XD_txt_dev.append(csr_to_rowview_torch(csr_T(DW_bm25), dev))

degD_tag = np.diff(DT_ppmi.indptr)
degD_txt = np.diff(DW_bm25.indptr)
start_tag = np.where(degD_tag > 0)[0].astype(np.int64)
start_txt = np.where(degD_txt > 0)[0].astype(np.int64)

RW_WALKS_PER_DOC    = int(rw_params["RW_WALKS_PER_DOC"])
RW_L_DOCS_PER_SENT  = int(rw_params["RW_L_DOCS_PER_SENT"])
RW_SEED_BASE        = int(rw_params["RW_SEED_BASE"])
RW_AVOID_BACKTRACK  = bool(rw_params["RW_AVOID_BACKTRACK"])
RW_RESTART_PROB     = float(rw_params["RW_RESTART_PROB"])
RW_X_DEGREE_POW     = float(rw_params["RW_X_DEGREE_POW"])
RW_X_NO_REPEAT_LAST = int(rw_params["RW_X_NO_REPEAT_LAST"])

def row_neighbors(indptr, indices, data, r: torch.Tensor):
    a = indptr[r]; b = indptr[r+1]
    if (b-a).item() <= 0: return None, None
    sl = slice(a.item(), b.item())
    return indices[sl], data[sl]

def sample_pos_by_weights(w: torch.Tensor, g: torch.Generator) -> int:
    if w is None or w.numel()==0: return -1
    w = torch.clamp(w, min=0)
    s = w.sum()
    if not torch.isfinite(s) or s.item()<=0: return -1
    cdf = torch.cumsum(w, dim=0)
    u = torch.rand((), generator=g, device=w.device) * cdf[-1]
    pos = torch.searchsorted(cdf, u, right=False).item()
    return min(pos, cdf.numel()-1)

class TorchWalkCorpus:
    def __init__(self, starts_np, DX_views, XD_views, base_seed, split_shards=32, view_name=""):
        self.starts_np = starts_np
        self.DX_views  = DX_views
        self.XD_views  = XD_views
        self.base_seed = base_seed
        self.split_shards = max(1, split_shards)
        self._iters = 0; self.view_name=view_name
    def __len__(self):
        return int(len(self.starts_np) * RW_WALKS_PER_DOC)
    def __iter__(self):
        self._iters += 1
        rng = np.random.default_rng(RW_SEED_BASE + 31*self._iters)
        starts = self.starts_np.copy(); rng.shuffle(starts)
        shards = np.array_split(starts, self.split_shards)
        devs = devices
        for shard_id, shard in enumerate(shards):
            dev = devs[shard_id % len(devs)]
            DX  = self.DX_views[shard_id % len(devs)]
            XD  = self.XD_views[shard_id % len(devs)]
            indptr_D, indices_D, data_D = DX
            indptr_X, indices_X, data_X = XD
            g = torch.Generator(device=dev)
            g.manual_seed(self.base_seed + 7919*(self._iters + shard_id))
            x_factor = None
            if abs(RW_X_DEGREE_POW) > 1e-12:
                x_deg = (indptr_X[1:]-indptr_X[:-1]).to(torch.float32)
                x_factor = torch.clamp(x_deg, min=1.0).pow(RW_X_DEGREE_POW)
            for d0 in shard:
                for _ in range(RW_WALKS_PER_DOC):
                    seq = [int(d0)]
                    prev_d=None; cur_d=int(d0); last_x=-1
                    for _step in range(RW_L_DOCS_PER_SENT-1):
                        r = torch.tensor(cur_d, dtype=torch.long, device=dev)
                        x_cols, x_w = row_neighbors(indptr_D, indices_D, data_D, r)
                        if x_cols is None: break
                        w = x_w.clone()
                        if x_factor is not None: w = w * x_factor[x_cols]
                        if RW_X_NO_REPEAT_LAST>0 and last_x>=0:
                            mask = (x_cols==last_x)
                            if mask.any(): w[mask]=0.0
                        pos_x = sample_pos_by_weights(w, g)
                        if pos_x < 0: break
                        x = int(x_cols[pos_x].item())
                        xr = torch.tensor(x, dtype=torch.long, device=dev)
                        d_rows, d_w = row_neighbors(indptr_X, indices_X, data_X, xr)
                        if d_rows is None: break
                        if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.numel()>1:
                            m = (d_rows==prev_d)
                            if m.any(): d_w=d_w.clone(); d_w[m]=0.0
                        pos_d = sample_pos_by_weights(d_w, g)
                        if pos_d < 0: break
                        next_d = int(d_rows[pos_d].item())
                        if torch.rand((), generator=g, device=dev).item() < RW_RESTART_PROB:
                            next_d = int(d0)
                        seq.append(next_d)
                        prev_d, cur_d, last_x = cur_d, next_d, x
                    if len(seq)>=2: yield [str(s) for s in seq]

tag_corpus  = TorchWalkCorpus(start_tag, DX_tag_dev, XD_tag_dev, base_seed=RW_SEED_BASE+11, split_shards=32, view_name="tag")
text_corpus = TorchWalkCorpus(start_txt, DX_txt_dev, XD_txt_dev, base_seed=RW_SEED_BASE+23, split_shards=32, view_name="text")

# ---------- 2) SGNS 模型 ----------
class SGNS(nn.Module):
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.in_emb  = nn.Embedding(vocab_size, dim, sparse=False)
        self.out_emb = nn.Embedding(vocab_size, dim, sparse=False)
        nn.init.uniform_(self.in_emb.weight,  -0.5/dim, 0.5/dim)
        nn.init.uniform_(self.out_emb.weight, -0.5/dim, 0.5/dim)
    def forward(self, center, pos, neg):
        v = self.in_emb(center)             # [B,d]
        u = self.out_emb(pos)               # [B,d]
        pos_logit = torch.sum(v*u, dim=1)   # [B]
        neg_u = self.out_emb(neg)           # [B,K,d]
        neg_logit = torch.einsum("bd,bkd->bk", v, neg_u)   # [B,K]
        pos_loss = torch.nn.functional.softplus(-pos_logit)
        neg_loss = torch.nn.functional.softplus(neg_logit).sum(dim=1)
        return (pos_loss + neg_loss).mean().unsqueeze(0)   # 返回 [1]，避免 DP 警告

def use_all_gpus(module: nn.Module) -> nn.Module:
    if torch.cuda.is_available() and USE_ALL_GPUS and torch.cuda.device_count()>1:
        return nn.DataParallel(module)
    return module

# ---------- 3) 负采样分布 ----------
def build_ns_dist_from_deg(deg: np.ndarray, power=0.75):
    p = np.power(np.maximum(deg, 1), power).astype(np.float64)
    p = p / p.sum()
    return p

ns_dist_tag  = build_ns_dist_from_deg(degD_tag, power=NS_POWER)
ns_dist_text = build_ns_dist_from_deg(degD_txt, power=NS_POWER)

# ---------- 4) 语料 → 正样本对（上下文子采样版） ----------
def iter_pairs_from_corpus(corpus, window: int, max_sents: int = None, seed: int = 0):
    rng = random.Random(seed)
    sent_count = 0
    ctx_samples = SGNS_CTX_SAMPLES if SGNS_CTX_SAMPLES is not None else None
    for sent in corpus:
        if max_sents is not None and sent_count >= max_sents: break
        s = [int(x) for x in sent]
        L = len(s)
        for i in range(L):
            l = max(0, i - window); r = min(L - 1, i + window)
            if l == r: continue
            if ctx_samples is None:
                for j in range(l, r + 1):
                    if j == i: continue
                    yield s[i], s[j]
            else:
                cand = list(range(l, i)) + list(range(i+1, r+1))
                if not cand: continue
                k = min(ctx_samples, len(cand))
                pick = rng.sample(cand, k)
                for j in pick:
                    yield s[i], s[j]
        sent_count += 1

# ---------- 5) 批处理 & GPU 负采样 ----------
def batch_pairs_and_negs(pair_iter, batch_size_pairs: int, negK: int,
                         ns_dist: np.ndarray, device: torch.device):
    centers, contexts = [], []
    ns_prob_t = None; ns_gen = None
    if NEG_SAMPLING_DEVICE == "gpu" and device.type == "cuda":
        ns_prob_t = torch.tensor(ns_dist, dtype=torch.float32, device=device)
        ns_prob_t = ns_prob_t / ns_prob_t.sum()
        ns_gen = torch.Generator(device=device)
        ns_gen.manual_seed(RNG_BASE_SEED + 1337)

    def to_device_1d(a: list):
        t = torch.tensor(a, dtype=torch.long)
        if torch.cuda.is_available():
            t = t.pin_memory()
        return t.to(device, non_blocking=True)

    for c, x in pair_iter:
        centers.append(c); contexts.append(x)
        if len(centers) >= batch_size_pairs:
            B = len(centers)
            centers_t = to_device_1d(centers)
            contexts_t= to_device_1d(contexts)
            if ns_prob_t is not None:
                neg_flat = torch.multinomial(ns_prob_t, num_samples=B*negK, replacement=True, generator=ns_gen)
                negs_t   = neg_flat.view(B, negK)
            else:
                negs = np.random.choice(len(ns_dist), size=(B, negK), p=ns_dist)
                negs_cpu = torch.tensor(negs, dtype=torch.long)
                if torch.cuda.is_available():
                    negs_cpu = negs_cpu.pin_memory()
                negs_t = negs_cpu.to(device, non_blocking=True)
            yield centers_t, contexts_t, negs_t
            centers.clear(); contexts.clear()

    if centers:
        B = len(centers)
        centers_t = to_device_1d(centers)
        contexts_t= to_device_1d(contexts)
        if ns_prob_t is not None:
            neg_flat = torch.multinomial(ns_prob_t, num_samples=B*negK, replacement=True, generator=ns_gen)
            negs_t   = neg_flat.view(B, negK)
        else:
            negs = np.random.choice(len(ns_dist), size=(B, negK), p=ns_dist)
            negs_cpu = torch.tensor(negs, dtype=torch.long)
            if torch.cuda.is_available():
                negs_cpu = negs_cpu.pin_memory()
            negs_t = negs_cpu.to(device, non_blocking=True)
        yield centers_t, contexts_t, negs_t

# ---------- 6) 训练单视图 ----------
def train_view(view_name: str, degD: np.ndarray, ns_dist: np.ndarray, corpus, max_sents: int, out_path: Path):
    torch.manual_seed(RNG_BASE_SEED + (11 if view_name=="tag" else 23))
    base_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = SGNS(vocab_size=N, dim=SGNS_DIM)
    model = use_all_gpus(model).to(base_device)
    optimizer = torch.optim.SGD(model.parameters(), lr=SGNS_LR)

    print(f"[Train-{view_name}] epochs={SGNS_EPOCHS}, dim={SGNS_DIM}, window={SGNS_WINDOW}, neg={SGNS_NEGATIVE}, batch_pairs={PAIRS_BATCH_SIZE}, accum={ACCUM_STEPS}")
    for ep in range(1, SGNS_EPOCHS+1):
        t0 = time.time()
        pair_iter = iter_pairs_from_corpus(corpus=corpus, window=SGNS_WINDOW,
                                           max_sents=max_sents, seed=RNG_BASE_SEED + ep)

        total_pairs = 0
        total_loss = 0.0
        model.train()
        optimizer.zero_grad(set_to_none=True)
        step_in_epoch = 0

        amp_dtype = None
        if SGNS_MIXED_PRECISION == "bf16":
            amp_dtype = torch.bfloat16
        elif SGNS_MIXED_PRECISION == "fp16":
            amp_dtype = torch.float16

        for centers_t, contexts_t, negs_t in batch_pairs_and_negs(
                pair_iter, PAIRS_BATCH_SIZE, SGNS_NEGATIVE, ns_dist, device=base_device):

            if amp_dtype is not None and base_device.type=='cuda':
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    loss = model(centers_t, contexts_t, negs_t)
                    if hasattr(loss, "dim") and loss.dim() != 0:
                        loss = loss.mean()
                    loss = loss / ACCUM_STEPS
            else:
                loss = model(centers_t, contexts_t, negs_t)
                if hasattr(loss, "dim") and loss.dim() != 0:
                    loss = loss.mean()
                loss = loss / ACCUM_STEPS

            loss.backward()
            step_in_epoch += 1
            if step_in_epoch % ACCUM_STEPS == 0:
                if SGNS_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), SGNS_CLIP_NORM)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            total_pairs += centers_t.size(0)
            total_loss  += float(loss.detach().item()) * centers_t.size(0) * ACCUM_STEPS

        # epoch 尾梯度
        if step_in_epoch % ACCUM_STEPS != 0:
            if SGNS_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), SGNS_CLIP_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        dt = time.time() - t0
        if total_pairs == 0:
            print(f"[Train-{view_name}] epoch {ep}: no pairs produced")
        else:
            print(f"[Train-{view_name}] epoch {ep}: pairs={total_pairs:,}, avg_loss={total_loss/total_pairs:.4f}, time={dt:.1f}s")

    # 导出向量（in_emb）、L2 归一并落盘
    if isinstance(model, nn.DataParallel):
        E = model.module.in_emb.weight.detach().cpu().numpy()
    else:
        E = model.in_emb.weight.detach().cpu().numpy()
    Z = E.astype(np.float32, copy=True)
    nrm = np.linalg.norm(Z, axis=1, keepdims=True)
    mask = (nrm[:,0] > 0)
    Z[mask] = Z[mask] / nrm[mask]

    emb_df = pd.DataFrame(Z, columns=[f"f{i}" for i in range(Z.shape[1])])
    emb_df.insert(0, "doc_idx", np.arange(N, dtype=np.int64))
    emb_df.to_parquet(out_path, engine="fastparquet", index=False)

    covered = int(mask.sum())
    print(f"[Train-{view_name}] saved {out_path.name}; covered_docs={covered}/{N} ({covered/max(N,1):.1%})")

    # 显存回收
    del model, optimizer; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------- 7) 顺序训练两视图（避免峰值叠加） ----------
train_view("tag",  degD_tag,  ns_dist_tag,  tag_corpus,  MAX_SENTS_PER_EPOCH_TAG,  TMP_DIR / "Z_tag.parquet")
train_view("text", degD_txt,  ns_dist_text, text_corpus, MAX_SENTS_PER_EPOCH_TEXT, TMP_DIR / "Z_text.parquet")

print("[Step 6] Done. Embeddings saved to Z_tag.parquet / Z_text.parquet")


[SGNS] using devices: ['cuda:0', 'cuda:1']
[Train-tag] epochs=3, dim=256, window=5, neg=12, batch_pairs=245760, accum=2


KeyboardInterrupt: 